# Wind Turbine Blade Panorama Stitching - Visualization

Pipeline: SAM Segmentation → LoFTR Matching → RANSAC Filtering → Coarse-to-Fine Fallback Transform

Visualize intermediate outputs for a specified DIU ID and section.

In [ ]:
import os
import sys
import json
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# Add blade_stitching to path for module imports
REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Load Models

In [ ]:
from modules.segmentation import load_sam, segment_image
from modules.matching import load_loftr, match_loftr, filter_by_mask, ransac_filter
from modules.brightness import align_brightness
from modules.coarse import compute_coarse_transforms, compute_transforms
from modules.stitching import stitch_trans_scale

WEIGHTS_DIR = REPO_DIR / 'weights'
load_sam(
    base_checkpoint=str(WEIGHTS_DIR / 'sam_vit_b_01ec64.pth'),
    finetune_checkpoint=str(WEIGHTS_DIR / 'best_model.pth'),
    device=device,
)
load_loftr(device=device)


## 2. Configuration

In [ ]:
# ── Change these to select what to visualize ──
DIU_ID = '40013'
SECTION_NAME = 'C-PS'  # e.g. 'A-LE', 'B-PS', 'C-TE', etc.

DATA_DIR = REPO_DIR / 'data'


## 3. Load Data

In [ ]:
draft_dir = DATA_DIR / DIU_ID
metadata_path = draft_dir / 'metadata.json'

with open(metadata_path, 'r') as f:
    metadata = json.load(f)
photos_by_id = {p['id']: p for p in metadata['photos']}

# Group photos by blade-side section
sections = defaultdict(list)
for p in metadata['photos']:
    key = f"{p['blade_tag']}-{p['blade_side_tag']}"
    sections[key].append(p['id'])

print(f"Draft ID: {metadata['draft_id']}")
print(f"Available sections: {list(sections.keys())}")
print(f"Selected: {SECTION_NAME} ({len(sections[SECTION_NAME])} photos)")


## 4. Process Images

In [ ]:
# Load images and distances
section_photo_ids = sections[SECTION_NAME]
images = []
distances = []
for pid in section_photo_ids:
    photo = photos_by_id[pid]
    local_path = photo['local_path']
    img_path = os.path.join(draft_dir, local_path)
    if not os.path.exists(img_path):
        img_path = os.path.join(draft_dir, photo['blade_tag'], photo['blade_side_tag'], os.path.basename(local_path))
    img = cv2.imread(img_path)
    if img is not None:
        images.append(img)
        distances.append(photo['metadata']['measured_distance_to_blade'])
print(f"Loaded {len(images)} images")

# Brightness alignment
images = align_brightness(images)
print("Brightness aligned")

# Generate masks
masks = [segment_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)) for img in images]
print(f"Generated {len(masks)} masks")

# Create segmented images for matching
images_segmented = [img.copy() for img in images]
for img, mask in zip(images_segmented, masks):
    img[mask == 0] = 0
print("Segmented images ready")


In [ ]:
# Match keypoints with LoFTR and apply RANSAC filtering
match_results = []
filtered_results = []

for i in range(len(images_segmented) - 1):
    pts1, pts2 = match_loftr(images_segmented[i], images_segmented[i+1])
    pts1, pts2 = filter_by_mask(pts1, pts2, masks[i], masks[i+1])
    match_results.append({'pts1': pts1, 'pts2': pts2})
    
    mask_ransac = ransac_filter(pts1, pts2) if len(pts1) >= 4 else np.ones(len(pts1), dtype=bool)
    filtered_results.append({'pts1': pts1[mask_ransac], 'pts2': pts2[mask_ransac]})
    print(f"Pair {i}: {len(pts1)} matches -> {mask_ransac.sum()} after RANSAC")


## 5. Visualize Keypoint Matches

In [ ]:
def draw_matches(img1, img2, pts1, pts2, max_matches=100):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1
    canvas[:h2, w1:w1+w2] = img2
    
    n_show = min(len(pts1), max_matches)
    indices = np.random.choice(len(pts1), n_show, replace=False) if len(pts1) > max_matches else range(len(pts1))
    
    for idx in indices:
        pt1 = (int(pts1[idx][0]), int(pts1[idx][1]))
        pt2 = (int(pts2[idx][0]) + w1, int(pts2[idx][1]))
        color = tuple(np.random.randint(100, 255, 3).tolist())
        cv2.line(canvas, pt1, pt2, color, 1)
        cv2.circle(canvas, pt1, 3, color, -1)
        cv2.circle(canvas, pt2, 3, color, -1)
    return canvas

# Visualize matches for all pairs
n_pairs = len(images_segmented) - 1
fig, axes = plt.subplots(n_pairs, 1, figsize=(16, 4*n_pairs))
if n_pairs == 1:
    axes = [axes]

for i in range(n_pairs):
    pts1 = filtered_results[i]['pts1']
    pts2 = filtered_results[i]['pts2']
    
    match_img = draw_matches(images_segmented[i], images_segmented[i+1], pts1, pts2)
    axes[i].imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Pair {i}: {len(pts1)} matches after RANSAC", fontsize=10)
    axes[i].axis('off')

plt.suptitle(f'{DIU_ID} / {SECTION_NAME}: LoFTR + RANSAC Matches', fontsize=14)
plt.tight_layout()
plt.show()


## 6. Compare Coarse vs Fine vs Fallback (Per Pair)

In [ ]:
# Compute coarse transforms using DCM
coarse_transforms = compute_coarse_transforms(photos_by_id, section_photo_ids, img_width=720, img_height=480)
print(f"Computed {len(coarse_transforms)} coarse transforms")

# Compute all transform modes
print("\nCoarse mode:")
transforms_coarse, decisions_coarse = compute_transforms(filtered_results, coarse_transforms, distances, masks=masks, mode='coarse')
print("\nFine mode:")
transforms_fine, decisions_fine = compute_transforms(filtered_results, coarse_transforms, distances, masks=masks, mode='fine')
print("\nFallback mode:")
transforms_fallback, decisions_fallback = compute_transforms(filtered_results, coarse_transforms, distances, masks=masks, mode='fallback')


In [ ]:
def stitch_pair_with_contour_and_iou(img1, img2, mask1, mask2, transform):
    """Stitch two images with mask contours and compute IoU in overlapping bbox."""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    tx, ty, scale = transform['tx'], transform['ty'], transform['scale']
    new_w2, new_h2 = int(w2 * scale), int(h2 * scale)
    if new_w2 <= 0 or new_h2 <= 0:
        return img1, 0.0

    img2_scaled = cv2.resize(img2, (new_w2, new_h2))
    mask2_scaled = cv2.resize(mask2.astype(np.uint8), (new_w2, new_h2), interpolation=cv2.INTER_NEAREST)

    min_x, min_y = min(0, tx), min(0, ty)
    max_x, max_y = max(w1, tx + new_w2), max(h1, ty + new_h2)
    canvas_w, canvas_h = int(max_x - min_x), int(max_y - min_y)
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

    x1_off, y1_off = int(-min_x), int(-min_y)
    x2_off, y2_off = int(tx - min_x), int(ty - min_y)
    canvas[y1_off:y1_off+h1, x1_off:x1_off+w1] = img1

    y2_end = min(y2_off + new_h2, canvas_h)
    x2_end = min(x2_off + new_w2, canvas_w)
    y2_start, x2_start = max(0, y2_off), max(0, x2_off)
    src_y, src_x = y2_start - y2_off, x2_start - x2_off

    if y2_end > y2_start and x2_end > x2_start:
        roi = img2_scaled[src_y:src_y+(y2_end-y2_start), src_x:src_x+(x2_end-x2_start)]
        img_mask = np.any(roi > 0, axis=2)
        canvas[y2_start:y2_end, x2_start:x2_end][img_mask] = roi[img_mask]

    # Mask regions for IoU
    mask1_region = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    mask1_region[y1_off:y1_off+h1, x1_off:x1_off+w1] = mask1
    mask2_region = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    if y2_end > y2_start and x2_end > x2_start:
        mask2_region[y2_start:y2_end, x2_start:x2_end] = mask2_scaled[src_y:src_y+(y2_end-y2_start), src_x:src_x+(x2_end-x2_start)]

    # IoU in overlapping bbox
    bx1, by1 = max(x1_off, x2_off), max(y1_off, y2_off)
    bx2, by2 = min(x1_off + w1, x2_off + new_w2), min(y1_off + h1, y2_off + new_h2)
    if bx2 > bx1 and by2 > by1:
        m1c, m2c = mask1_region[by1:by2, bx1:bx2], mask2_region[by1:by2, bx1:bx2]
        inter = np.logical_and(m1c > 0, m2c > 0).sum()
        union = np.logical_or(m1c > 0, m2c > 0).sum()
        iou = inter / union if union > 0 else 0.0
    else:
        iou = 0.0

    # Draw image edges and mask contours
    cv2.rectangle(canvas, (x1_off, y1_off), (x1_off + w1 - 1, y1_off + h1 - 1), (0, 200, 0), 1)
    cv2.rectangle(canvas, (x2_off, y2_off), (x2_off + new_w2 - 1, y2_off + new_h2 - 1), (0, 0, 200), 1)
    contours1, _ = cv2.findContours(mask1_region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, contours1, -1, (0, 255, 0), 2)
    contours2, _ = cv2.findContours(mask2_region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, contours2, -1, (0, 0, 255), 2)

    return canvas, iou

# Compare all three methods for each pair
n_pairs = len(images) - 1
fig, axes = plt.subplots(n_pairs, 3, figsize=(20, 5*n_pairs))
if n_pairs == 1:
    axes = axes.reshape(1, -1)

iou_data_fine = []

for i in range(n_pairs):
    coarse_t = transforms_coarse[i]
    fine_t = transforms_fine[i]
    fallback_t = transforms_fallback[i]

    d_c = np.sqrt(coarse_t['tx']**2 + coarse_t['ty']**2)
    d_f = np.sqrt(fine_t['tx']**2 + fine_t['ty']**2)
    d_fb = np.sqrt(fallback_t['tx']**2 + fallback_t['ty']**2)

    id1 = section_photo_ids[i]
    id2 = section_photo_ids[i + 1]

    stitched_coarse, iou_c = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], coarse_t)
    stitched_fine, iou_f = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], fine_t)
    stitched_fallback, iou_fb = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], fallback_t)

    iou_data_fine.append({'pair': i, 'iou': iou_f})

    axes[i, 0].imshow(cv2.cvtColor(stitched_coarse, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Pair {i} [{id1}-{id2}] COARSE: d={d_c:.1f}, IoU={iou_c:.2f}", fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(cv2.cvtColor(stitched_fine, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"Pair {i} [{id1}-{id2}] FINE: d={d_f:.1f}, IoU={iou_f:.2f}", fontsize=10)
    axes[i, 1].axis('off')

    fb_color = 'green' if decisions_fallback[i] == 'fine' else 'orange'
    axes[i, 2].imshow(cv2.cvtColor(stitched_fallback, cv2.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"Pair {i} [{id1}-{id2}] FALLBACK [{decisions_fallback[i].upper()}]: d={d_fb:.1f}, IoU={iou_fb:.2f}", fontsize=10, color=fb_color)
    axes[i, 2].axis('off')

plt.suptitle(f'{DIU_ID} / {SECTION_NAME}: Coarse vs Fine vs Fallback\n(Thin rect=image edge, Thick=mask contour)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Print sorted by IoU (increasing order) - FINE mode
print("\n" + "="*60)
print("FINE MODE: SORTED BY IoU - INCREASING ORDER")
print("="*60)
print(f"{'Pair':<6} {'IoU':>10}")
print("-" * 20)

sorted_data = sorted(iou_data_fine, key=lambda x: x['iou'])
for d in sorted_data:
    print(f"{d['pair']:<6} {d['iou']:>10.3f}")

print(f"\nFallback Summary: {decisions_fallback.count('fine')} FINE, {decisions_fallback.count('coarse')} COARSE")


## 7. Edge Alignment

In [ ]:
from modules.edge_alignment import compute_edge_aligned_transforms

print("Computing edge-aligned transforms from fallback...")
edge_aligned_transforms, alignment_results = compute_edge_aligned_transforms(images, masks, transforms_fallback)


In [ ]:
## Edge Detection Debug: step-by-step visualization per pair
from modules.edge_alignment import (
    detect_lines_from_mask, compute_line_angle, compute_line_length,
    group_lines_by_angle, select_best_group, group_lines_by_position,
    select_line_from_edge_group, clip_line_to_bbox, line_intersection,
    find_perpendicular_line_crossing_all, compute_group_mean_angle, angle_distance,
)

n_pairs = len(images) - 1
angle_tolerance = 15

for pair_idx in range(n_pairs):
    img1, img2 = images[pair_idx], images[pair_idx + 1]
    mask1, mask2 = masks[pair_idx], masks[pair_idx + 1]
    transform = transforms_fallback[pair_idx]

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    tx, ty, scale = transform['tx'], transform['ty'], transform['scale']
    new_w2, new_h2 = int(w2 * scale), int(h2 * scale)

    if new_w2 <= 0 or new_h2 <= 0:
        print(f"Pair {pair_idx}: FAILED at scaling (new_w2={new_w2}, new_h2={new_h2})")
        continue

    mask2_scaled = cv2.resize(mask2.astype(np.uint8), (new_w2, new_h2), interpolation=cv2.INTER_NEAREST)
    min_x, min_y = min(0, tx), min(0, ty)
    max_x, max_y = max(w1, tx + new_w2), max(h1, ty + new_h2)
    canvas_w, canvas_h = int(max_x - min_x), int(max_y - min_y)
    x1_offset, y1_offset = int(-min_x), int(-min_y)
    x2_offset, y2_offset = int(tx - min_x), int(ty - min_y)

    mask1_region = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    mask1_region[y1_offset:y1_offset+h1, x1_offset:x1_offset+w1] = mask1
    mask2_region = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    y2_end = min(y2_offset + new_h2, canvas_h)
    x2_end = min(x2_offset + new_w2, canvas_w)
    y2_start, x2_start = max(0, y2_offset), max(0, x2_offset)
    if y2_end > y2_start and x2_end > x2_start:
        mask2_region[y2_start:y2_end, x2_start:x2_end] = mask2_scaled[
            y2_start-y2_offset:y2_end-y2_offset, x2_start-x2_offset:x2_end-x2_offset]

    bbox = (max(x1_offset, x2_offset), max(y1_offset, y2_offset),
            min(x1_offset + w1, x2_offset + new_w2), min(y1_offset + h1, y2_offset + new_h2))

    if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        print(f"Pair {pair_idx}: FAILED - no overlap bbox")
        continue

    center1 = (x1_offset + w1 / 2, y1_offset + h1 / 2)
    center2 = (x2_offset + new_w2 / 2, y2_offset + new_h2 / 2)
    img2_bounds = (x2_offset, y2_offset, x2_offset + new_w2, y2_offset + new_h2)

    # Step 1: detect lines in overlap
    mask1_overlap = mask1_region[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    mask2_overlap = mask2_region[bbox[1]:bbox[3], bbox[0]:bbox[2]]
    raw_lines1 = detect_lines_from_mask(mask1_overlap)
    raw_lines2 = detect_lines_from_mask(mask2_overlap)
    # shift to canvas coords
    lines1 = [[bbox[0]+x1, bbox[1]+y1, bbox[0]+x2, bbox[1]+y2] for x1, y1, x2, y2 in raw_lines1]
    lines2 = [[bbox[0]+x1, bbox[1]+y1, bbox[0]+x2, bbox[1]+y2] for x1, y1, x2, y2 in raw_lines2]

    # Step 2: group by angle
    groups1 = group_lines_by_angle(lines1, angle_tolerance)
    groups2 = group_lines_by_angle(lines2, angle_tolerance)
    selected1, sel_idx1 = select_best_group(groups1)
    selected2, sel_idx2 = select_best_group(groups2)

    # Step 3: group by position (two edges)
    edge1_1, edge2_1 = group_lines_by_position(selected1)
    edge1_2, edge2_2 = group_lines_by_position(selected2)

    # Step 4: select best line per edge
    fl1_a = select_line_from_edge_group(edge1_1, center1, center2)
    fl1_b = select_line_from_edge_group(edge2_1, center1, center2)
    fl2_a = select_line_from_edge_group(edge1_2, center1, center2)
    fl2_b = select_line_from_edge_group(edge2_2, center1, center2)
    final_lines1 = [l for l in [fl1_a, fl1_b] if l is not None]
    final_lines2 = [l for l in [fl2_a, fl2_b] if l is not None]

    # Step 5: angle check
    angle_ok = False
    perp_line = None
    intersections1, intersections2 = [], []
    if len(final_lines1) == 2 and len(final_lines2) == 2:
        all_lines = final_lines1 + final_lines2
        angles = [compute_line_angle(l) for l in all_lines]
        mean_angle = compute_group_mean_angle(angles)
        angle_ok = all(angle_distance(a, mean_angle) <= angle_tolerance for a in angles)

        if angle_ok:
            # Step 6: perpendicular line
            perp_angle = (mean_angle + 90) % 180
            perp_line, _, _ = find_perpendicular_line_crossing_all(all_lines, perp_angle, bbox, img2_bounds)

            if perp_line is not None:
                # Step 7: intersections
                clipped1 = [clip_line_to_bbox(l, bbox) for l in final_lines1]
                clipped2 = [clip_line_to_bbox(l, bbox) for l in final_lines2]
                clipped1 = [l for l in clipped1 if l]
                clipped2 = [l for l in clipped2 if l]
                tol = 2.0
                def _find_ints(lines):
                    pts = []
                    for edge_line in lines:
                        pt = line_intersection(perp_line, edge_line)
                        if pt:
                            x1, y1, x2, y2 = edge_line
                            if (min(x1,x2)-tol <= pt[0] <= max(x1,x2)+tol and
                                min(y1,y2)-tol <= pt[1] <= max(y1,y2)+tol):
                                pts.append(pt)
                    return pts
                intersections1 = _find_ints(clipped1) if len(clipped1) == 2 else []
                intersections2 = _find_ints(clipped2) if len(clipped2) == 2 else []

    # ── Determine failure step ──
    if len(lines1) == 0 or len(lines2) == 0:
        fail_step = "Step 1: line detection"
    elif len(selected1) == 0 or len(selected2) == 0:
        fail_step = "Step 2: angle grouping"
    elif len(final_lines1) != 2 or len(final_lines2) != 2:
        fail_step = "Step 3-4: edge selection"
        detail_parts = []
        if len(final_lines1) != 2: detail_parts.append(f"img1 got {len(final_lines1)} lines")
        if len(final_lines2) != 2: detail_parts.append(f"img2 got {len(final_lines2)} lines")
        fail_step += f" ({', '.join(detail_parts)})"
    elif not angle_ok:
        fail_step = "Step 5: angle alignment"
    elif perp_line is None:
        fail_step = "Step 6: perpendicular line"
    elif len(intersections1) != 2 or len(intersections2) != 2:
        fail_step = f"Step 7: intersections (img1={len(intersections1)}, img2={len(intersections2)})"
    else:
        fail_step = None  # success

    # ── Draw 6-column visualization ──
    # Cols: [overlap masks] [all detected lines] [best angle group] [2 edges + final lines] [perp + intersections] [status]
    fig, axes = plt.subplots(1, 6, figsize=(30, 4))
    fig.suptitle(f"Pair {pair_idx}: Edge Detection {'OK' if fail_step is None else 'FAILED → ' + fail_step}",
                 fontsize=12, color='green' if fail_step is None else 'red')

    # Helper: make base canvas with masks
    def make_canvas():
        c = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
        c[mask1_region > 0] = [60, 60, 60]
        c[mask2_region > 0] = np.maximum(c[mask2_region > 0], [40, 40, 40]) + [0, 0, 40]
        # draw bbox
        cv2.rectangle(c, (bbox[0], bbox[1]), (bbox[2]-1, bbox[3]-1), (80, 80, 80), 1)
        return c

    colors_img1 = (0, 255, 0)  # green for img1
    colors_img2 = (0, 100, 255)  # blue for img2

    # Col 0: overlap region masks
    c = make_canvas()
    # highlight overlap
    overlap = np.logical_and(mask1_region > 0, mask2_region > 0)
    c[overlap] = [100, 100, 0]
    contours1, _ = cv2.findContours(mask1_overlap, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours1:
        cnt[:, :, 0] += bbox[0]; cnt[:, :, 1] += bbox[1]
        cv2.drawContours(c, [cnt], -1, colors_img1, 1)
    contours2, _ = cv2.findContours(mask2_overlap, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours2:
        cnt[:, :, 0] += bbox[0]; cnt[:, :, 1] += bbox[1]
        cv2.drawContours(c, [cnt], -1, colors_img2, 1)
    axes[0].imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Overlap masks", fontsize=9)
    axes[0].axis('off')

    # Col 1: all detected lines
    c = make_canvas()
    for l in lines1:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img1, 1)
    for l in lines2:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img2, 1)
    axes[1].imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"All lines (img1={len(lines1)}, img2={len(lines2)})", fontsize=9)
    axes[1].axis('off')

    # Col 2: best angle group
    c = make_canvas()
    for item in selected1:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img1, 2)
    for item in selected2:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img2, 2)
    n_grp1 = len(groups1)
    n_grp2 = len(groups2)
    axes[2].imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"Best group (grps: {n_grp1},{n_grp2} | sel: {len(selected1)},{len(selected2)})", fontsize=9)
    axes[2].axis('off')

    # Col 3: position groups + final selected lines
    c = make_canvas()
    # Draw edge groups with different shades
    for item in edge1_1:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 180, 0), 1)
    for item in edge2_1:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 255, 0), 1)
    for item in edge1_2:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 80, 180), 1)
    for item in edge2_2:
        l = item[0]
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 120, 255), 1)
    # Draw final selected lines thick
    for l in final_lines1:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 255, 0), 3)
    for l in final_lines2:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), (0, 120, 255), 3)
    axes[3].imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    e1 = f"{len(edge1_1)}+{len(edge2_1)}"
    e2 = f"{len(edge1_2)}+{len(edge2_2)}"
    axes[3].set_title(f"Edges ({e1} | {e2}), final={len(final_lines1)},{len(final_lines2)}", fontsize=9)
    axes[3].axis('off')

    # Col 4: perpendicular line + intersections
    c = make_canvas()
    for l in final_lines1:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img1, 2)
    for l in final_lines2:
        cv2.line(c, (int(l[0]), int(l[1])), (int(l[2]), int(l[3])), colors_img2, 2)
    if perp_line is not None:
        cv2.line(c, (int(perp_line[0]), int(perp_line[1])), (int(perp_line[2]), int(perp_line[3])), (0, 255, 255), 2)
    for pt in intersections1:
        cv2.circle(c, (int(pt[0]), int(pt[1])), 5, (0, 255, 0), -1)
    for pt in intersections2:
        cv2.circle(c, (int(pt[0]), int(pt[1])), 5, (0, 100, 255), -1)
    perp_status = "found" if perp_line is not None else "NONE"
    axes[4].imshow(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
    axes[4].set_title(f"Perp: {perp_status}, ints={len(intersections1)},{len(intersections2)}", fontsize=9)
    axes[4].axis('off')

    # Col 5: summary text
    axes[5].axis('off')
    summary_lines = [
        f"Pair {pair_idx}",
        f"",
        f"1. Lines: {len(lines1)}, {len(lines2)}",
        f"2. Groups: {n_grp1}, {n_grp2}",
        f"   Best: {len(selected1)}, {len(selected2)}",
        f"3. Edges: {e1} | {e2}",
        f"4. Final: {len(final_lines1)}, {len(final_lines2)}",
    ]
    if len(final_lines1) == 2 and len(final_lines2) == 2:
        angles_str = ", ".join(f"{a:.0f}" for a in angles)
        summary_lines.append(f"5. Angles: [{angles_str}]")
        summary_lines.append(f"   Aligned: {angle_ok}")
    if perp_line is not None:
        summary_lines.append(f"6. Perp line: OK")
    elif len(final_lines1) == 2 and len(final_lines2) == 2 and angle_ok:
        summary_lines.append(f"6. Perp line: FAILED")
    summary_lines.append(f"7. Intersections: {len(intersections1)}, {len(intersections2)}")
    summary_lines.append(f"")
    if fail_step is None:
        summary_lines.append(f"RESULT: OK")
    else:
        summary_lines.append(f"FAILED: {fail_step}")

    axes[5].text(0.05, 0.95, "\n".join(summary_lines), transform=axes[5].transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace',
                 color='green' if fail_step is None else 'red')

    plt.tight_layout()
    plt.show()


In [ ]:
# Visualize edge alignment process per pair
n_pairs = len(images) - 1
fig, axes = plt.subplots(n_pairs, 4, figsize=(24, 4*n_pairs))
if n_pairs == 1:
    axes = axes.reshape(1, -1)

for i, result in enumerate(alignment_results):
    fb_t = result['fallback_t']
    trans_t = result['translated_t']
    scaled_t = result['scaled_t']
    final_t = result['final_t']

    iou_fb = result['iou_fallback']
    iou_trans = result['iou_translated']
    iou_scaled = result['iou_scaled']
    trans_applied = result['translation_applied']
    scale_applied = result['scaling_applied']

    # Compute scale factor from alignment data
    sf = None
    if result['alignment_data'] is not None:
        x_1, x_2 = np.array(result['alignment_data']['x_1']), np.array(result['alignment_data']['x_2'])
        y_1, y_2 = np.array(result['alignment_data']['y_1']), np.array(result['alignment_data']['y_2'])
        d_1 = np.linalg.norm(x_2 - x_1)
        d_2 = np.linalg.norm(y_2 - y_1)
        sf = d_1 / d_2 if d_2 > 0 else None

    # Column 1: Fallback
    canvas_fb, _ = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], fb_t)
    axes[i, 0].imshow(cv2.cvtColor(canvas_fb, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Pair {i}: Fallback (IoU={iou_fb:.3f})", fontsize=10)
    axes[i, 0].axis('off')

    # Column 2: +Translation
    if trans_t is not None:
        canvas_trans, _ = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], trans_t)
        axes[i, 1].imshow(cv2.cvtColor(canvas_trans, cv2.COLOR_BGR2RGB))
        diff = iou_trans - iou_fb
        color = 'green' if trans_applied else 'red'
        status = 'APPLIED' if trans_applied else 'SKIPPED'
        axes[i, 1].set_title(f"+Translation (IoU={iou_trans:.3f}, {'+' if diff>=0 else ''}{diff:.3f}) [{status}]", fontsize=9, color=color)
        for spine in axes[i, 1].spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(3)
    else:
        axes[i, 1].text(0.5, 0.5, f"Edge detection failed\n{result['reason']}", ha='center', va='center', fontsize=11, color='red', transform=axes[i, 1].transAxes)
        axes[i, 1].set_title("+Translation [FAILED]", fontsize=9, color='red')
    axes[i, 1].axis('off')

    # Column 3: +Scaling (show scale factor)
    if scaled_t is not None and trans_applied:
        canvas_scaled, _ = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], scaled_t)
        axes[i, 2].imshow(cv2.cvtColor(canvas_scaled, cv2.COLOR_BGR2RGB))
        diff = iou_scaled - iou_trans
        color = 'green' if scale_applied else 'red'
        status = 'APPLIED' if scale_applied else 'SKIPPED'
        sf_str = f" sf={sf:.4f}" if sf is not None else ""
        axes[i, 2].set_title(f"+Scaling{sf_str} (IoU={iou_scaled:.3f}, {'+' if diff>=0 else ''}{diff:.3f}) [{status}]", fontsize=9, color=color)
        for spine in axes[i, 2].spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(3)
    elif not trans_applied:
        sf_str = f" sf={sf:.4f}" if sf is not None else ""
        canvas_fb2, _ = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], fb_t)
        axes[i, 2].imshow(cv2.cvtColor(canvas_fb2, cv2.COLOR_BGR2RGB))
        axes[i, 2].set_title(f"+Scaling{sf_str} [N/A - translation skipped]", fontsize=9, color='gray')
    else:
        axes[i, 2].text(0.5, 0.5, "Not computed", ha='center', va='center', fontsize=11, color='gray', transform=axes[i, 2].transAxes)
        axes[i, 2].set_title("+Scaling [N/A]", fontsize=9, color='gray')
    axes[i, 2].axis('off')

    # Column 4: Final result
    canvas_final, iou_final = stitch_pair_with_contour_and_iou(images[i], images[i+1], masks[i], masks[i+1], final_t)
    axes[i, 3].imshow(cv2.cvtColor(canvas_final, cv2.COLOR_BGR2RGB))
    if trans_applied and scale_applied:
        final_label = "Trans+Scale"
    elif trans_applied:
        final_label = "Trans only"
    else:
        final_label = "Fallback"
    axes[i, 3].set_title(f"Final: {final_label} (IoU={iou_final:.3f})", fontsize=10)
    axes[i, 3].axis('off')

plt.suptitle(f'{DIU_ID} / {SECTION_NAME}: Edge Alignment (Green=applied, Red=skipped)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Edge alignment summary
print(f"\nEdge Alignment Summary for {DIU_ID} / {SECTION_NAME}:")
print(f"{'Pair':<6} {'Fallback':<10} {'Trans':<10} {'Scaled':<10} {'SF':<10} {'Final':<15} {'Reason':<25}")
print("-" * 90)
for result in alignment_results:
    i = result['pair']
    iou_fb = result['iou_fallback']
    iou_trans = result['iou_translated'] if result['translated_t'] else 0
    iou_scaled = result['iou_scaled'] if result['scaled_t'] else 0

    # Compute scale factor
    sf_str = ""
    if result['alignment_data'] is not None:
        x_1, x_2 = np.array(result['alignment_data']['x_1']), np.array(result['alignment_data']['x_2'])
        y_1, y_2 = np.array(result['alignment_data']['y_1']), np.array(result['alignment_data']['y_2'])
        d_1 = np.linalg.norm(x_2 - x_1)
        d_2 = np.linalg.norm(y_2 - y_1)
        sf = d_1 / d_2 if d_2 > 0 else 0
        sf_str = f"{sf:.4f}"

    if result['scaling_applied']:
        final = "Trans+Scale"
    elif result['translation_applied']:
        final = "Trans only"
    else:
        final = "Fallback"

    reason = result['reason'] if result['reason'] else "OK"
    print(f"{i:<6} {iou_fb:<10.4f} {iou_trans:<10.4f} {iou_scaled:<10.4f} {sf_str:<10} {final:<15} {reason:<25}")

n_ts = sum(1 for r in alignment_results if r['scaling_applied'])
n_to = sum(1 for r in alignment_results if r['translation_applied'] and not r['scaling_applied'])
n_fb = sum(1 for r in alignment_results if not r['translation_applied'])
print("-" * 90)
print(f"Trans+Scale: {n_ts}, Trans only: {n_to}, Fallback: {n_fb}")


## 8. Panorama Comparison

In [ ]:
from modules.stitching import stitch_trans_scale

panorama_fallback = stitch_trans_scale(images, transforms_fallback)
panorama_edge = stitch_trans_scale(images, edge_aligned_transforms)

fig, axes = plt.subplots(1, 2, figsize=(24, 12))
axes[0].imshow(cv2.cvtColor(panorama_fallback, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Fallback ({panorama_fallback.shape[1]}x{panorama_fallback.shape[0]})', fontsize=14)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(panorama_edge, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Edge Aligned ({panorama_edge.shape[1]}x{panorama_edge.shape[0]})', fontsize=14)
axes[1].axis('off')
plt.suptitle(f'{DIU_ID} / {SECTION_NAME}: Fallback vs Edge Aligned', fontsize=16)
plt.tight_layout()
plt.show()


## 9. Summary

In [ ]:
print("="*60)
print(f"SUMMARY: {DIU_ID} / {SECTION_NAME}")
print("="*60)
print(f"  Images:          {len(images)}")
print(f"  Pairs:           {len(images) - 1}")
print(f"  Fallback:        {decisions_fallback.count('fine')} fine, {decisions_fallback.count('coarse')} coarse")
print(f"  Edge alignment:  {n_ts} trans+scale, {n_to} trans only, {n_fb} fallback")
print(f"  Panorama (fb):   {panorama_fallback.shape}")
print(f"  Panorama (edge): {panorama_edge.shape}")
